In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
import os
import datasets
import logging
import json
import threading
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.vectorstores import VectorStore
from smolagents import OpenAIServerModel, CodeAgent, Tool
from smolagents.monitoring import LogLevel
from transformers import AutoTokenizer
from tqdm import tqdm
from typing import List, Optional
from utils.blablador import Models


In [ ]:
## old vector creator
def sanitize_filename(dataset_name):
    """Convert dataset name to a valid filename by replacing '/' with '_'"""
    return dataset_name.replace("/", "_")


def load_or_create_vectordb(dataset_name):
    """Load existing vectordb or create new one if it doesn't exist"""

    # Create sanitized filename
    safe_filename = sanitize_filename(dataset_name)
    vectordb_path = os.path.join("vectordb", safe_filename)

    # Check if vectordb already exists
    if os.path.exists(vectordb_path):
        print(f"Found existing vector database at {vectordb_path}")
        print("Loading existing vector database...")

        # Load the embedding model (needed for loading)
        embedding_model = HuggingFaceEmbeddings(
            model_name="thenlper/gte-small")

        # Load the existing vectordb
        vectordb = FAISS.load_local(vectordb_path,
                                    embedding_model,
                                    allow_dangerous_deserialization=True)
        print("Vector database loaded successfully!")
        return vectordb

    else:
        print(f"No existing vector database found at {vectordb_path}")
        print("Creating new vector database...")

        # Load and process the dataset
        knowledge_base = datasets.load_dataset(dataset_name, split="train")
        source_docs = [
            Document(page_content=doc["text"],
                     metadata={"source": doc["source"].split("/")[1]})
            for doc in knowledge_base
        ]

        # Initialize text splitter
        text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
            AutoTokenizer.from_pretrained("thenlper/gte-small"),
            chunk_size=200,
            chunk_overlap=20,
            add_start_index=True,
            strip_whitespace=True,
            separators=["\n\n", "\n", ".", " ", ""],
        )

        # Split docs and keep only unique ones
        print("Splitting documents...")
        docs_processed = []
        unique_texts = {}
        for doc in tqdm(source_docs):
            new_docs = text_splitter.split_documents([doc])
            for new_doc in new_docs:
                if new_doc.page_content not in unique_texts:
                    unique_texts[new_doc.page_content] = True
                    docs_processed.append(new_doc)

        print("Embedding documents...")
        embedding_model = HuggingFaceEmbeddings(
            model_name="thenlper/gte-small")
        vectordb = FAISS.from_documents(
            documents=docs_processed,
            embedding=embedding_model,
            distance_strategy=DistanceStrategy.COSINE,
        )

        # Save the vectordb
        print(f"Saving vector database to {vectordb_path}...")
        vectordb.save_local(vectordb_path)
        print("Vector database saved successfully!")

        return vectordb


# Usage
dataset_name = "m-ric/huggingface_doc"
vectordb = load_or_create_vectordb(dataset_name)

print(f"Vector database ready with {vectordb.index.ntotal} vectors!")

In [3]:
## Load or create vector database parallelly


def sanitize_filename(dataset_name: str) -> str:
    """Convert dataset name to a valid filename by replacing '/' with '_'"""
    return dataset_name.replace("/", "_")


class DocumentProcessor:
    """Thread-safe document processor to avoid pickling issues"""

    def __init__(self, chunk_size: int = 200, chunk_overlap: int = 20):
        self.text_splitter = None
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self._lock = threading.Lock()

    def _get_text_splitter(self):
        """Initialize text splitter in thread-local manner"""
        if self.text_splitter is None:
            with self._lock:
                if self.text_splitter is None:
                    self.text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
                        AutoTokenizer.from_pretrained("thenlper/gte-small"),
                        chunk_size=self.chunk_size,
                        chunk_overlap=self.chunk_overlap,
                        add_start_index=True,
                        strip_whitespace=True,
                        separators=["\n\n", "\n", ".", " ", ""],
                    )
        return self.text_splitter

    def split_documents_chunk(self,
                              docs_chunk: List[Document]) -> List[Document]:
        """Split a chunk of documents using the text splitter"""
        text_splitter = self._get_text_splitter()
        processed_docs = []
        for doc in docs_chunk:
            new_docs = text_splitter.split_documents([doc])
            processed_docs.extend(new_docs)
        return processed_docs


def parallel_document_splitting(
        source_docs: List[Document],
        max_workers: int = None,
        chunk_size: int = 100,
        text_chunk_size: int = 200,
        text_chunk_overlap: int = 20) -> List[Document]:
    """Split documents in parallel using ThreadPoolExecutor"""
    if max_workers is None:
        max_workers = min(8, len(source_docs) // chunk_size + 1)

    print(f"Splitting documents in parallel using {max_workers} threads...")

    # Split source_docs into chunks for parallel processing
    doc_chunks = [
        source_docs[i:i + chunk_size]
        for i in range(0, len(source_docs), chunk_size)
    ]

    # Create a shared processor instance with configurable chunk size
    processor = DocumentProcessor(text_chunk_size, text_chunk_overlap)

    # Use ThreadPoolExecutor instead of ProcessPoolExecutor
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Process chunks in parallel with progress bar
        results = list(
            tqdm(executor.map(processor.split_documents_chunk, doc_chunks),
                 total=len(doc_chunks),
                 desc="Processing document chunks"))

    # Flatten results
    all_processed_docs = []
    for chunk_result in results:
        all_processed_docs.extend(chunk_result)

    return all_processed_docs


def remove_duplicates(docs: List[Document]) -> List[Document]:
    """Remove duplicate documents based on page_content"""
    print("Removing duplicate documents...")
    unique_texts = set()
    unique_docs = []

    for doc in tqdm(docs, desc="Checking for duplicates"):
        if doc.page_content not in unique_texts:
            unique_texts.add(doc.page_content)
            unique_docs.append(doc)

    print(f"Removed {len(docs) - len(unique_docs)} duplicate documents")
    return unique_docs


def batch_embed_documents(docs: List[Document],
                          embedding_model,
                          batch_size: int = 100) -> FAISS:
    """Create FAISS vectorstore with batch processing for embeddings"""
    print(f"Embedding documents in batches of {batch_size}...")

    if not docs:
        raise ValueError("No documents to embed")

    # Create initial vectorstore with first batch
    first_batch = docs[:batch_size]
    print(f"Creating initial vectorstore with {len(first_batch)} documents...")

    vectordb = FAISS.from_documents(
        documents=first_batch,
        embedding=embedding_model,
        distance_strategy=DistanceStrategy.COSINE,
    )

    # Process remaining documents in batches
    remaining_docs = docs[batch_size:]
    if remaining_docs:
        total_batches = (len(remaining_docs) + batch_size - 1) // batch_size

        for i in tqdm(range(0, len(remaining_docs), batch_size),
                      desc="Adding document batches",
                      total=total_batches):
            batch = remaining_docs[i:i + batch_size]

            # Create temporary vectorstore for this batch
            batch_vectordb = FAISS.from_documents(
                documents=batch,
                embedding=embedding_model,
                distance_strategy=DistanceStrategy.COSINE,
            )

            # Merge with main vectorstore
            vectordb.merge_from(batch_vectordb)

    return vectordb


def sequential_document_splitting(
        source_docs: List[Document],
        text_chunk_size: int = 200,  # Add this parameter
        text_chunk_overlap: int = 20) -> List[Document]:
    """Fallback sequential document splitting if parallel processing fails"""
    print("Using sequential document splitting...")

    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        AutoTokenizer.from_pretrained("thenlper/gte-small"),
        chunk_size=text_chunk_size,
        chunk_overlap=text_chunk_overlap,
        add_start_index=True,
        strip_whitespace=True,
        separators=["\n\n", "\n", ".", " ", ""],
    )

    docs_processed = []
    for doc in tqdm(source_docs, desc="Splitting documents"):
        new_docs = text_splitter.split_documents([doc])
        docs_processed.extend(new_docs)

    return docs_processed


def load_or_create_vectordb(dataset_name: str,
                            batch_size: int = 100,
                            max_workers: int = None,
                            doc_chunk_size: int = 100,
                            text_chunk_size: int = 200,
                            text_chunk_overlap: int = 20,
                            force_rebuild: bool = False,
                            use_parallel: bool = True) -> FAISS:
    """Load existing vectordb or create new one with optimized processing"""

    # Create sanitized filename
    safe_filename = sanitize_filename(dataset_name)
    vectordb_path = os.path.join(
        "vectordb",
        f"{safe_filename}_{text_chunk_size}")  # Include chunk size in filename

    # Check if vectordb already exists and not forcing rebuild
    if os.path.exists(vectordb_path) and not force_rebuild:
        print(f"Found existing vector database at {vectordb_path}")
        print("Loading existing vector database...")

        try:
            # Load the embedding model (needed for loading)
            embedding_model = HuggingFaceEmbeddings(
                model_name="thenlper/gte-small")

            # Load the existing vectordb
            vectordb = FAISS.load_local(vectordb_path,
                                        embedding_model,
                                        allow_dangerous_deserialization=True)
            print("Vector database loaded successfully!")
            return vectordb

        except Exception as e:
            print(f"Error loading existing vectordb: {e}")
            print("Creating new vector database...")

    else:
        if force_rebuild:
            print("Force rebuild requested. Creating new vector database...")
        else:
            print(f"No existing vector database found at {vectordb_path}")
            print("Creating new vector database...")

    # Load and process the dataset
    print("Loading dataset...")
    knowledge_base = datasets.load_dataset(dataset_name, split="train")
    source_docs = [
        Document(page_content=doc["text"],
                 metadata={"source": doc["source"].split("/")[1]})
        for doc in tqdm(knowledge_base, desc="Processing dataset")
    ]

    print(f"Loaded {len(source_docs)} documents from dataset")

    # Split documents (with fallback to sequential processing)
    try:
        if use_parallel and len(source_docs) > 50:
            docs_processed = parallel_document_splitting(
                source_docs, max_workers, doc_chunk_size, text_chunk_size,
                text_chunk_overlap)
        else:
            docs_processed = sequential_document_splitting(
                source_docs, text_chunk_size, text_chunk_overlap)

    except Exception as e:
        print(f"Parallel processing failed: {e}")
        print("Falling back to sequential processing...")
        docs_processed = sequential_document_splitting(
            source_docs, text_chunk_size,
            text_chunk_overlap)  # Pass new parameters

    print(f"Split into {len(docs_processed)} document chunks")

    # Remove duplicates
    docs_processed = remove_duplicates(docs_processed)

    print(f"Final document count after deduplication: {len(docs_processed)}")

    # Create embedding model
    print("Loading embedding model...")
    embedding_model = HuggingFaceEmbeddings(model_name="thenlper/gte-small")

    # Create vectordb with batch processing
    vectordb = batch_embed_documents(docs_processed, embedding_model,
                                     batch_size)

    # Save the vectordb
    print(f"Saving vector database to {vectordb_path}...")
    try:
        vectordb.save_local(vectordb_path)
        print("Vector database saved successfully!")
    except Exception as e:
        print(f"Error saving vectordb: {e}")
        print("Continuing without saving...")

    return vectordb


# Usage example with configurable parameters
dataset_name = "m-ric/huggingface_doc"

# Usage example with configurable chunk sizes
config = {
    "batch_size": 50,
    "max_workers": 4,
    "doc_chunk_size": 100,  # Parallel processing batch size
    "text_chunk_size": 400,  # Document content chunk size (tokens) 
    "text_chunk_overlap": 40,  # Overlap between chunks
    "force_rebuild": False,
    "use_parallel": True
}

vectordb = load_or_create_vectordb(dataset_name, **config)


Found existing vector database at vectordb\m-ric_huggingface_doc_400
Loading existing vector database...


C:\Users\Philippe\AppData\Local\Temp\ipykernel_13148\866353223.py:185: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Vector database loaded successfully!


In [9]:
# Example of using the vectordb for similarity search
query = "How to use transformers?"
results = vectordb.similarity_search(query, k=5)
for i, result in enumerate(results):
    print(f"Result {i+1}: {result.page_content}")

Result 1: . We can use this model as we would any other Transformers model, for instance by loading it in a pipeline. Try the push_to_hub API on your next training to easily share your model with the rest of the world!
Result 2: How do Transformers work?[[how-do-transformers-work]]

<CourseFloatingBanner
    chapter={1}
    classNames="absolute z-10 right-0 top-0"
/>

In this section, we will take a high-level look at the architecture of Transformer models.

## A bit of Transformer history[[a-bit-of-transformer-history]]

Here are some reference points in the (short) history of Transformer models:

<div class="flex justify-center">
<img class="block dark:hidden" src="https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter1/transformers_chrono.svg" alt="A brief chronology of Transformers models.">
<img class="hidden dark:block" src="https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter1/transformers_chrono

In [4]:
class RetrieverTool(Tool):
    name = "retriever"
    description = """
    Retrieves relevant documents from the knowledge base using semantic similarity search.
    Choose k based on query complexity:
    - k=5-7: For standard questions needing some context  
    - k=8-10: For complex topics requiring multiple perspectives
    - k=10-13: For comprehensive research or broad exploration
    """

    # inputs = {
    #     "query": {
    #         "type":
    #         "string",
    #         "description":
    #         "The search query. Use descriptive keywords and phrases rather than questions. Example: 'machine learning algorithms' instead of 'what are ML algorithms?'",
    #     },
    #     "k": {
    #         "type": "integer",
    #         "description":
    #         "Number of documents to retrieve (3-10). Choose based on query complexity: 3 for standard questions, 5-7 for complex topics, 8-10 for comprehensive research. Default: 3 if not specified.",
    #         "nullable": True
    #     }
    # }
    inputs = {
        "query": {
            "type":
            "string",
            "description":
            "The search query. Use descriptive keywords and phrases rather than questions. Example: 'machine learning algorithms' instead of 'what are ML algorithms?'",
        },
        "k": {
            "type": "integer",
            "description":
            "Number of documents to retrieve (7-20). Start with 7, increase if no relevant information is found. Default: 7 if not specified.",
            "nullable": True
        }
    }
    output_type = "string"

    def __init__(self,
                 vectordb: VectorStore,
                 k: int = 7,
                 score_threshold: Optional[float] = None,
                 max_content_length: int = 10000,
                 **kwargs):
        super().__init__(**kwargs)
        self.vectordb = vectordb
        self.k = k
        self.score_threshold = score_threshold
        self.max_content_length = max_content_length
        self.logger = logging.getLogger(self.__class__.__name__)

    def forward(self, query: str, k: Optional[int] = None) -> str:
        # Input validation
        if not query or not isinstance(query, str) or not query.strip():
            return "Error: Query must be a non-empty string"

        # Parameter validation
        retrieval_k = k if k is not None else self.k
        if retrieval_k is not None and (not isinstance(retrieval_k, int) or
                                        retrieval_k < 3 or retrieval_k > 20):
            retrieval_k = self.k

        query = query.strip()

        try:
            # Try to get documents with scores if available
            if hasattr(self.vectordb, 'similarity_search_with_score'
                       ) and self.score_threshold is not None:
                docs_with_scores = self.vectordb.similarity_search_with_score(
                    query, k=retrieval_k)
                # Filter by score threshold
                filtered_docs = [
                    doc for doc, score in docs_with_scores
                    if score >= self.score_threshold
                ]
                docs = filtered_docs[:retrieval_k] if filtered_docs else []

                if not docs and docs_with_scores:
                    # If no docs meet threshold, take the best one anyway
                    docs = [docs_with_scores[0][0]]

            else:
                docs = self.vectordb.similarity_search(query, k=retrieval_k)

        except Exception as e:
            self.logger.error(f"Error during similarity search: {str(e)}")
            return f"Error retrieving documents: {str(e)}"

        # Handle no results
        if not docs:
            return f"No relevant documents found for query: '{query}'"

        # Format results
        return self._format_results(query, docs)

    def _format_results(self, query: str, docs) -> str:
        """Format the retrieved documents into a readable string."""
        result_parts = [
            f"Search Query: {query}",
            f"Retrieved {len(docs)} relevant document(s):", ""
        ]

        for i, doc in enumerate(docs, 1):
            # Add document header
            result_parts.append(f"=== Document {i} ===")

            # Add metadata if available
            if hasattr(doc, 'metadata') and doc.metadata:
                metadata_info = []
                if 'source' in doc.metadata:
                    metadata_info.append(f"Source: {doc.metadata['source']}")
                if 'title' in doc.metadata:
                    metadata_info.append(f"Title: {doc.metadata['title']}")
                if 'page' in doc.metadata:
                    metadata_info.append(f"Page: {doc.metadata['page']}")

                if metadata_info:
                    result_parts.append(" | ".join(metadata_info))

            # Add content (with truncation if needed)
            content = doc.page_content
            if len(content) > self.max_content_length:
                content = content[:self.max_content_length].rsplit(
                    ' ', 1)[0] + "... [truncated]"

            result_parts.append(content)
            result_parts.append("")  # Empty line between documents

        return "\n".join(result_parts).strip()

    def get_retrieval_stats(self) -> dict:
        """Get basic stats about the vector store if available."""
        try:
            # This depends on the vector store implementation
            if hasattr(self.vectordb, '__len__'):
                return {"total_documents": len(self.vectordb)}
            return {"status": "Vector store connected"}
        except Exception:
            return {"status": "Unable to retrieve stats"}

In [6]:
# old retriever tool
from smolagents import Tool
from langchain_core.vectorstores import VectorStore


class RetrieverTool(Tool):
    name = "retriever"
    description = "Using semantic similarity, retrieves some documents from the knowledge base that have the closest embeddings to the input query."
    inputs = {
        "query": {
            "type":
            "string",
            "description":
            "The query to perform. This should be semantically close to your target documents. Use the affirmative form rather than a question.",
        }
    }
    output_type = "string"

    def __init__(self, vectordb: VectorStore, **kwargs):
        super().__init__(**kwargs)
        self.vectordb = vectordb

    def forward(self, query: str) -> str:
        assert isinstance(query, str), "Your search query must be a string"

        docs = self.vectordb.similarity_search(
            query,
            k=7,
        )

        return "\nRetrieved documents:\n" + "".join([
            f"===== Document {str(i)} =====\n" + doc.page_content
            for i, doc in enumerate(docs)
        ])

In [61]:
# answer LLM model initialization

load_dotenv()
# model_name = "gemini-2.5-flash-preview-05-20"
# model_name = "gemini-1.5-flash"

# answer_llm = OpenAIServerModel(
#     model_id=model_name,
#     api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
#     api_key=os.getenv("Gemini_API_KEY2"),
#     temperature=0.2)

models = Models(api_key=os.getenv("Blablador_API_KEY")).get_model_ids()
model_id_blablador = 4
model_name = models[model_id_blablador].split(" - ")[1]
print("The adgent uses the following model:", model_name)
answer_llm = OpenAIServerModel(
    model_id=models[model_id_blablador],
    api_base="https://helmholtz-blablador.fz-juelich.de:8000/v1",
    api_key=os.getenv("Blablador_API_KEY"),
    flatten_messages_as_text=True,
    temperature=0.2)
retriever_tool = RetrieverTool(vectordb)
# retriever_tool = RetrieverTool(vectordb, score_threshold=0.7)

agent = CodeAgent(
    tools=[retriever_tool],
    model=answer_llm,
    planning_interval=2,
    max_steps=30,
    verbosity_level=LogLevel.ERROR,
)

The adgent uses the following model: QwenLong L1 32B


In [ ]:
agent_output = agent.run("How can I push a model to the Hub?")

print("Final output:")
print(agent_output)

In [ ]:
inspect = eval_dataset.select([4])

print(inspect[0]["context"])

!--Copyright 2022 The HuggingFace Team. All rights reserved.

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with
the License. You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on
an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the
specific language governing permissions and limitations under the License.

⚠️ Note that this file is in Markdown but contain specific syntax for our doc-builder (similar to MDX) that may not be
rendered properly in your Markdown viewer.

-->

# LongT5

## Overview

The LongT5 model was proposed in [LongT5: Efficient Text-To-Text Transformer for Long Sequences](https://arxiv.org/abs/2112.07916)
by Mandy Guo, Joshua Ainslie, David Uthus, Santiago Ontanon, Jianmo Ni, Yun-Hsuan Sung and Yinf

In [13]:
retriever_tool = RetrieverTool(vectordb, max_content_length=float('inf'))
print(
    retriever_tool(
        "The LongT5 model was proposed in [LongT5: Efficient Text-To-Text Transformer for Long Sequences]",
        k=3))

Search Query: The LongT5 model was proposed in [LongT5: Efficient Text-To-Text Transformer for Long Sequences]
Retrieved 3 relevant document(s):

=== Document 1 ===
Source: transformers
!--Copyright 2022 The HuggingFace Team. All rights reserved.

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with
the License. You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on
an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the
specific language governing permissions and limitations under the License.

⚠️ Note that this file is in Markdown but contain specific syntax for our doc-builder (similar to MDX) that may not be
rendered properly in your Markdown viewer.

-->

# LongT5

## Overview

The LongT5 model was proposed in [LongT5

In [6]:
eval_dataset = datasets.load_dataset("m-ric/huggingface_doc_qa_eval",
                                     split="train")

In [ ]:
##### Using agentic RAG to answer questions

outputs_agentic_rag = []

inspect = eval_dataset.select([20])
# inspect = eval_dataset.select([42, 43, 47, 49])
# inspect = eval_dataset.select([20])

for example in tqdm(inspect):
    question = example["question"]

    #     enhanced_question = f"""Using the information contained in your knowledge base, which you can access with the 'retriever' tool,
    # give a comprehensive answer to the question below.
    # Respond only to the question asked, response should be concise and relevant to the question.
    # If you cannot find information, do not give up and try calling your retriever again with different arguments!
    # Make sure to have covered the question completely by calling the retriever tool several times with semantically different queries.
    # Your queries should not be questions but affirmative form sentences: e.g. rather than "How do I load a model from the Hub in bf16?", query should be "load a model from the Hub bf16 weights".

    # Question:
    # {question}"""
    # enhanced_question = f"""Use the 'retriever' tool to find information and answer this question: {question}

    # Instructions:
    # - Search the knowledge base using relevant keywords (avoid using "hugging face" in search terms)
    # - Start with k=7, then try k=12, k=15, or k=20 if no relevant information is found
    # - Keep increasing the number of documents until you find the answer
    # - Try different search terms if initial results don't answer the question
    # - Don't stop until you have enough information to answer the question

    # """

    # enhanced_question = f"""Use the 'retriever' tool to find information and answer this question: {question}

    # REQUIRED SEARCH PATTERN:
    # 1. Search with k=7 using your best keywords
    # 2. If no relevant info found, search same keywords with k=10
    # 3. If still no relevant info, search same keywords with k=13
    # 4. If still no relevant info, try different keywords and repeat steps 1-3

    # SEARCH RULES:
    # - Use specific technical keywords
    # - Do NOT use "Hugging Face" in query
    # - Focus on the core concept/function/method being asked about
    # - Try different keyword combinations if first set doesn't work
    # - Don't stop until you find relevant information or exhaust keyword options

    # Start searching now with k=7.
    # """

    enhanced_question = f"""Use the 'retriever' tool to find information and answer this question: {question}

    SEARCH STRATEGY:
    1. Start with k=7 using your best keywords from the question
    2. If information is incomplete or irrelevant, retry with k=10 (same keywords)
    3. If the same documents are retrieved and information is still insufficient, try different keywords:
    - Use synonym words from the question to find different documents
    - Reduce keywords to search for broader documents
    - Use related concepts from previously retrieved documents
    4. Repeat steps 1-3 with new keyword combinations until you find the answer

    SEARCH RULES:
    - Do NOT include "Hugging Face" in queries (all questions relate to Hugging Face ecosystem)
    - The answer exists in the database - keep searching with different approaches
    - Try both specific and broad keyword strategies

    RESPONSE RULES:
    - Answer strictly based on retrieved documents, not prior knowledge
    - If answer is found: provide clear, direct response with relevant details
    - If answer not found: summarize the most relevant information retrieved

    Begin your search systematically.
    """

    answer = agent.run(enhanced_question)
    print("=======================================================")
    print(f"Question: {question}")
    print(f"Answer: {answer}")
    print(f'True answer: {example["answer"]}')

    results_agentic = {
        "question": question,
        "true_answer": example["answer"],
        "source_doc": example["source_doc"],
        "generated_answer": answer,
    }
    outputs_agentic_rag.append(results_agentic)
    # time.sleep(5)

In [121]:
with open("checkpoints/rag_checkpoint.json", 'r') as f:
    loaded_json = json.load(f)

# Convert back to a dictionary of DataFrames
# loaded_results = {}
# for system_type, records in loaded_json.items():
#     loaded_results[system_type] = pd.DataFrame.from_records(records)

In [62]:
## Set a checkpoint for query limitations (limited API) to use agentic RAG to answer questions


def run_agentic_rag_with_checkpointing(
        eval_dataset,
        agent,
        checkpoint_file="checkpoints/rag_checkpoint.json"):
    """
    Run agentic RAG with checkpointing to allow resuming from interruptions.
    
    Args:
        eval_dataset: Dataset containing questions and answers
        agent: The agent to run queries with
        checkpoint_file: File to store checkpointing information
    
    Returns:
        List of results from agentic RAG
    """
    outputs_agentic_rag = []
    start_idx = 0

    # Load checkpoint if exists
    if os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'r') as f:
                checkpoint_data = json.load(f)
                outputs_agentic_rag = checkpoint_data.get('results', [])
                start_idx = checkpoint_data.get('next_idx', 0)
                print(
                    f"Resuming from checkpoint at index {start_idx} with {len(outputs_agentic_rag)} results already processed"
                )
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting from beginning")

    # Create progress bar starting from checkpoint
    total_examples = len(eval_dataset)
    pbar = tqdm(total=total_examples, initial=start_idx)

    try:
        for idx in range(start_idx, total_examples):
            example = eval_dataset[idx]
            question = example["question"]

            enhanced_question = f"""Use the 'retriever' tool to find information and answer this question: {question}

            SEARCH STRATEGY:
            1. Start with k=7 using your best keywords from the question
            2. If information is incomplete or irrelevant, retry with k=10 (same keywords)
            3. If the same documents are retrieved and information is still insufficient, try different keywords:
            - Use synonym words from the question to find different documents
            - Reduce keywords to search for broader documents
            - Use related concepts from previously retrieved documents
            4. Repeat steps 1-3 with new keyword combinations until you find the answer

            SEARCH RULES:
            - Do NOT include "Hugging Face" in queries (all questions relate to Hugging Face ecosystem)
            - The answer exists in the database - keep searching with different approaches
            - Try both specific and broad keyword strategies

            RESPONSE RULES:
            - Answer strictly based on retrieved documents, not prior knowledge
            - If answer is found: provide clear, direct response with relevant details
            - If answer not found: summarize the most relevant information retrieved

            Begin your search systematically.
            """

            try:
                answer = agent.run(enhanced_question)

                print(
                    "=======================================================")
                print(f"Question: {question}")
                print(f"Answer: {answer}")
                print(f'True answer: {example["answer"]}')

                results_agentic = {
                    "question": question,
                    "true_answer": example["answer"],
                    "source_doc": example["source_doc"],
                    "generated_answer": answer,
                }

                outputs_agentic_rag.append(results_agentic)

                # Update checkpoint after each successful query
                save_checkpoint(checkpoint_file, outputs_agentic_rag, idx + 1)

                pbar.update(1)
                # time.sleep(10)  # Rate limiting

            except Exception as e:
                print(f"Error processing example {idx}: {e}")
                # Save checkpoint before exiting
                save_checkpoint(checkpoint_file, outputs_agentic_rag, idx)
                print(f"Checkpoint saved. Resume from index {idx}")
                raise

    except KeyboardInterrupt:
        print("\nProcess interrupted by user")
        # Save checkpoint before exiting
        save_checkpoint(checkpoint_file, outputs_agentic_rag, idx)
        print(f"Checkpoint saved. Resume from index {idx}")

    finally:
        pbar.close()

    return outputs_agentic_rag


def save_checkpoint(checkpoint_file, results, next_idx):
    """Save checkpoint data to file"""
    checkpoint_data = {
        "results": results,
        "next_idx": next_idx,
        "timestamp": time.time()
    }

    # Use a temporary file to avoid corruption if interrupted during writing
    temp_file = f"{checkpoint_file}.temp"
    with open(temp_file, 'w') as f:
        json.dump(checkpoint_data, f)

    # Safely replace the old checkpoint file
    if os.path.exists(checkpoint_file):
        os.replace(temp_file, checkpoint_file)
    else:
        os.rename(temp_file, checkpoint_file)


outputs_agentic_rag = run_agentic_rag_with_checkpointing(eval_dataset, agent)

  2%|▏         | 1/65 [00:09<09:39,  9.06s/it]

Question: What architecture is the `tokenizers-linux-x64-musl` binary designed for?

Answer: x86_64-unknown-linux-musl
True answer: x86_64-unknown-linux-musl


  3%|▎         | 2/65 [00:25<14:08, 13.47s/it]

Question: What is the purpose of the BLIP-Diffusion model?

Answer: BLIP-Diffusion enables zero-shot subject-driven generation and control-guided zero-shot generation for text-to-image tasks.
True answer: The BLIP-Diffusion model is designed for controllable text-to-image generation and editing.


  5%|▍         | 3/65 [00:45<16:44, 16.20s/it]

...

Question: What is the name of the open-source library created by Hugging Face to simplify Transformer acceleration?

Answer: Optimum
True answer: Optimum


 98%|█████████▊| 64/65 [19:50<00:20, 20.03s/it]

Question: What parameter is used to ensure that elements in a row have the same height in Gradio?

Answer: equal_height
True answer: equal_height


100%|██████████| 65/65 [20:23<00:00, 18.83s/it]

Question: What is the command to install the latest version of Optimum with OpenVINO support?

Answer: pip install --upgrade-strategy eager optimum[openvino,nncf]
True answer: pip install --upgrade-strategy eager optimum["openvino"]


In [27]:
##### using only RAG to answer questions

# answer_llm = OpenAIServerModel(
#     model_id=model_name,
#     api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
#     api_key=os.getenv("Gemini_API_KEY2"),
#     # flatten_messages_as_text=True,
#     temperature=0.2)

answer_llm = OpenAIServerModel(
    model_id=models[4],
    api_base="https://helmholtz-blablador.fz-juelich.de:8000/v1",
    api_key=os.getenv("Blablador_API_KEY"),
    # flatten_messages_as_text=True,
    temperature=0.2)

# inspect = eval_dataset.select(list(range(20)))

outputs_standard_rag = []

for example in tqdm(eval_dataset):
    question = example["question"]
    context = retriever_tool(question, k=7)

    prompt = f"""Given the question and supporting documents below, give a comprehensive answer to the question.
Respond only to the question asked, response should be concise and relevant to the question.
If the question is ambiguous or cannot be answered definitively, state so clearly.

Question:
{question}

{context}
"""
    messages = [{"role": "user", "content": prompt}]
    answer = answer_llm.generate(messages).content

    print("=======================================================")
    print(f"Question: {question}")
    # print(f"Context: {context}")
    print(f"Answer: {answer}")
    print(f'True answer: {example["answer"]}')

    results_agentic = {
        "question": question,
        "true_answer": example["answer"],
        "source_doc": example["source_doc"],
        "generated_answer": answer,
    }
    outputs_standard_rag.append(results_agentic)
    time.sleep(5)

  0%|          | 0/65 [00:00<?, ?it/s]

Question: What architecture is the `tokenizers-linux-x64-musl` binary designed for?

Answer: The `tokenizers-linux-x64-musl` binary is designed for the **x86_64 architecture** on Linux systems using the musl libc. This is explicitly stated in Document 1, which specifies the target as `x86_64-unknown-linux-musl`.
</think>
The `tokenizers-linux-x64-musl` binary is designed for the **x86_64 architecture** on Linux systems using the musl libc. This is explicitly stated in Document 1, which specifies the target as `x86_64-unknown-linux-musl`.
True answer: x86_64-unknown-linux-musl


  2%|▏         | 1/65 [00:07<08:03,  7.55s/it]

Question: What is the purpose of the BLIP-Diffusion model?

Answer: The BLIP-Diffusion model, as described in Document 2, enables zero-shot subject-driven generation and control-guided zero-shot generation. It allows users to generate images based on textual prompts while maintaining control over specific subjects within the generated images, leveraging pre-trained subject representations for improved controllability and relevance.
</think>
The BLIP-Diffusion model enables **zero-shot subject-driven generation** and **control-guided zero-shot generation**. It allows users to generate images based on textual prompts while maintaining control over specific subjects within the images, leveraging pre-trained subject representations for improved controllability and relevance.
True answer: The BLIP-Diffusion model is designed for controllable text-to-image generation and editing.


  3%|▎         | 2/65 [00:14<07:34,  7.21s/it]

...

 97%|█████████▋| 63/65 [08:24<00:15,  7.70s/it]

Question: What parameter is used to ensure that elements in a row have the same height in Gradio?

Answer: The parameter used to ensure that elements in a row have the same height in Gradio is `equal_height`. This parameter should be passed to the `.style()` method of `gr.Row()`, as mentioned in Document 1 (PR 3125).
</think>
The parameter used to ensure elements in a row have the same height in Gradio is **`equal_height`**. It should be passed to the `.style()` method of `gr.Row()`, as documented in PR 3125 (Document 1).
True answer: equal_height


 98%|█████████▊| 64/65 [08:31<00:07,  7.45s/it]

Question: What is the command to install the latest version of Optimum with OpenVINO support?

Answer: The command to install the latest version of Optimum with OpenVINO support is:

```bash
pip install --upgrade-strategy eager optimum[openvino,nncf]
```

This installs Optimum along with OpenVINO and NNCF dependencies for model optimization.
</think>
The command to install the latest version of Optimum with OpenVINO support is:

```bash
pip install --upgrade-strategy eager optimum[openvino,nncf]
```

This installs Optimum along with OpenVINO and NNCF dependencies for model optimization.
True answer: pip install --upgrade-strategy eager optimum["openvino"]


100%|██████████| 65/65 [08:38<00:00,  7.97s/it]


In [16]:
##### using vanilla LLM to answer questions

# answer_llm = OpenAIServerModel(
#     model_id=models[4],
#     api_base="https://helmholtz-blablador.fz-juelich.de:8000/v1",
#     api_key=Blablador_API_KEY,
#     # flatten_messages_as_text=True,
#     temperature=0.2)

outputs_standard = []

for example in tqdm(eval_dataset):
    question = example["question"]

    prompt = f"""Answer the following question as clearly and concisely as possible.
Use your own knowledge to respond.
If the question is ambiguous or cannot be answered definitively, state so clearly.

Question:
{question}

"""
    messages = [{"role": "user", "content": prompt}]
    answer = answer_llm.generate(messages).content

    print("=======================================================")
    print(f"Question: {question}")
    print(f"Answer: {answer}")
    print(f'True answer: {example["answer"]}')

    results_llm = {
        "question": question,
        "true_answer": example["answer"],
        "source_doc": example["source_doc"],
        "generated_answer": answer,
    }
    outputs_standard.append(results_llm)
    # time.sleep(5)

  2%|▏         | 1/65 [00:02<02:24,  2.26s/it]

Question: What architecture is the `tokenizers-linux-x64-musl` binary designed for?

Answer: The `tokenizers-linux-x64-musl` binary is designed for the **x86-64 (AMD64)** architecture on Linux systems. The "musl" suffix indicates it's compiled against the **musl libc** library, which is lightweight and commonly used in Alpine Linux and other minimal environments. This binary is optimized for compatibility with x64 processors running Linux distributions that use musl libc.
</think>
The `tokenizers-linux-x64-musl` binary is designed for the **x86-64 (AMD64)** architecture on Linux systems. The "musl" suffix indicates it's compiled against the **musl libc** library, used in environments like Alpine Linux.
True answer: x86_64-unknown-linux-musl


  3%|▎         | 2/65 [00:03<01:56,  1.84s/it]

Question: What is the purpose of the BLIP-Diffusion model?

Answer: The BLIP-Diffusion model is designed to generate high-quality images from text prompts by combining the capabilities of BLIP (a multimodal model for image-text understanding) with diffusion models. Its primary purpose is to enhance image generation by leveraging both text understanding and iterative noise reduction processes, enabling more accurate and context-aware image synthesis.
</think>
The BLIP-Diffusion model combines text understanding (via BLIP) with diffusion-based image generation to create high-quality images from text prompts, improving accuracy and contextual awareness in the output.
True answer: The BLIP-Diffusion model is designed for controllable text-to-image generation and editing.


  5%|▍         | 3/65 [00:07<02:55,  2.82s/it]

...

Question: What is the name of the open-source library created by Hugging Face to simplify Transformer acceleration?

Answer: The open-source library created by Hugging Face to simplify Transformer acceleration is **Transformers**. It provides tools and models for NLP tasks, optimized for performance and ease of use.
</think>
The open-source library created by Hugging Face to simplify Transformer acceleration is **Transformers**. It provides tools and models for NLP tasks, optimized for performance and ease of use.
</think>
The open-source library created by Hugging Face to simplify Transformer acceleration is **Transformers**. It provides tools and models for NLP tasks, optimized for performance and ease of use.
True answer: Optimum


 98%|█████████▊| 64/65 [02:37<00:01,  1.92s/it]

Question: What parameter is used to ensure that elements in a row have the same height in Gradio?

Answer: In Gradio, the parameter `equal_height` is used within the `Row` component to ensure that all elements in a row have the same height.
</think>
The parameter used in Gradio to ensure elements in a row have the same height is **`equal_height`**. This is specified within the `Row` component.
True answer: equal_height


100%|██████████| 65/65 [02:43<00:00,  2.51s/it]

Question: What is the command to install the latest version of Optimum with OpenVINO support?

Answer: The command to install the latest version of Optimum with OpenVINO support is:

```bash
pip install optimum[openvino]
```

This installs Optimum along with the OpenVINO integration package.
</think><think><think>
Okay, let's tackle this question. The user wants to know the command to install the latest version of Optimum with OpenVINO support. 

First, I need to recall how Optimum packages are structured. From what I remember, Optimum is a library by Hugging Face for optimizing machine learning models. They use extras for different backends. So, for OpenVINO support, there's probably an extra dependency group.

The standard way to install extras with pip is using square brackets. For example, package[extra]. So, the command should be pip install optimum[openvino]. That makes sense because OpenVINO is a backend they support.

I should also check if there are any version considerations.

In [17]:
EVALUATION_PROMPT = """You are a fair evaluator language model.

You will be given an instruction, a response to evaluate, a reference answer that gets a score of 3, and a score rubric representing a evaluation criteria are given.
1. Write a detailed feedback that assess the quality of the response strictly based on the given score rubric, not evaluating in general.
2. After writing a feedback, write a score that is an integer between 1 and 3. You should refer to the score rubric.
3. The output format should look as follows: \"Feedback: {{write a feedback for criteria}} [RESULT] {{an integer number between 1 and 3}}\"
4. Please do not generate any other opening, closing, and explanations. Be sure to include [RESULT] in your output.
5. Do not score conciseness: a correct answer that covers the question should receive max score, even if it contains additional useless information.

The instruction to evaluate:
{instruction}

Response to evaluate:
{response}

Reference Answer (Score 3):
{reference_answer}

Score Rubrics:
[Is the response complete, accurate, and factual based on the reference answer?]
Score 1: The response is completely incomplete, inaccurate, and/or not factual.
Score 2: The response is somewhat complete, accurate, and/or factual.
Score 3: The response is completely complete, accurate, and/or factual.

Feedback:"""

In [63]:
# evaluation

evaluation_llm_name = "gemini-2.0-flash"
evaluation_llm = OpenAIServerModel(
    model_id=evaluation_llm_name,
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("Gemini_API_KEY2"),
    temperature=0)

# results = {}
for system_type, outputs in [
    ("agentic_rag", outputs_agentic_rag),
        # ("standard_rag", outputs_standard_rag),
        # ("standard", outputs_standard),
]:
    print("evaluating {}".format(system_type))

    for experiment in tqdm(outputs):
        eval_prompt = EVALUATION_PROMPT.format(
            instruction=experiment["question"],
            response=experiment["generated_answer"],
            reference_answer=experiment["true_answer"],
        )
        messages = [
            {
                "role": "system",
                "content": "You are a fair evaluator language model."
            },
            {
                "role": "user",
                "content": eval_prompt
            },
        ]

        eval_result = evaluation_llm.generate(messages).content

        try:
            feedback, score = [
                item.strip() for item in eval_result.split("[RESULT]")
            ]
            experiment["eval_score_LLM_judge"] = score
            experiment["eval_feedback_LLM_judge"] = feedback
        except:
            print(f"Parsing failed - output was: {eval_result}")

        time.sleep(5)

    results[system_type] = pd.DataFrame.from_dict(outputs)
    results[system_type] = results[
        system_type].loc[~results[system_type]["generated_answer"].str.
                         contains("Error", na=False)]

evaluating agentic_rag


100%|██████████| 65/65 [06:16<00:00,  5.79s/it]


In [64]:
DEFAULT_SCORE = 2  # Give average score whenever scoring fails


def fill_score(x):
    try:
        return int(x)
    except:
        return DEFAULT_SCORE


for system_type, outputs in [
    ("agentic_rag", outputs_agentic_rag),
    ("standard_rag", outputs_standard_rag),
    ("standard", outputs_standard),
]:

    results[system_type]["eval_score_LLM_judge_int"] = (
        results[system_type]["eval_score_LLM_judge"].fillna(
            DEFAULT_SCORE).apply(fill_score))
    results[system_type]["eval_score_LLM_judge_int"] = (
        results[system_type]["eval_score_LLM_judge_int"] - 1) / 2

    print(
        f"Average score for {system_type} : {results[system_type]['eval_score_LLM_judge_int'].mean()*100:.1f}%"
    )

Average score for agentic_rag : 95.4%
Average score for standard_rag : 94.6%
Average score for standard : 60.8%


In [65]:
results["agentic_rag"]

,question,true_answer,source_doc,generated_answer,eval_score_LLM_judge,eval_feedback_LLM_judge,eval_score_LLM_judge_int
0,What architecture is the `tokenizers-linux-x64...,x86_64-unknown-linux-musl,huggingface/tokenizers/blob/main/bindings/node...,x86_64-unknown-linux-musl,3,"The response is completely complete, accurate,...",1.0
1,What is the purpose of the BLIP-Diffusion mode...,The BLIP-Diffusion model is designed for contr...,huggingface/diffusers/blob/main/docs/source/en...,BLIP-Diffusion enables zero-shot subject-drive...,3,The response accurately identifies the core fu...,1.0
2,How can a user claim authorship of a paper on ...,By clicking their name on the corresponding Pa...,huggingface/hub-docs/blob/main/docs/hub/paper-...,To claim authorship of a paper on the Hugging ...,3,The response provides a step-by-step guide tha...,1.0
3,What is the purpose of the /healthcheck endpoi...,Ensure the app is running,huggingface/datasets-server/blob/main/services...,The purpose of the /healthcheck endpoint in th...,3,The response accurately states that the /healt...,1.0
4,What is the default context window size for Lo...,127 tokens,huggingface/transformers/blob/main/docs/source...,127,3,The response accurately states the default con...,1.0
...,...,...,...,...,...,...,...
60,What is the maximum size of a model checkpoint...,10GB,huggingface/transformers/blob/main/docs/source...,10GB,3,The response accurately and completely answers...,1.0
61,What is the purpose of Weights and Biases (W&B...,To track their machine learning experiments at...,gradio-app/gradio/blob/main/guides/06_integrat...,Weights and Biases (W&B) is a tool designed fo...,3,The response accurately describes the purpose ...,1.0
62,What is the name of the open-source library cr...,Optimum,huggingface/blog/blob/main/intel.md,Optimum,3,"The response ""Optimum"" is a complete and accur...",1.0
63,What parameter is used to ensure that elements...,equal_height,gradio-app/gradio/blob/main/guides/cn/03_build...,equal_height,3,The response accurately identifies the paramet...,1.0


In [66]:
results["standard_rag"]

,question,true_answer,source_doc,generated_answer,eval_score_LLM_judge,eval_feedback_LLM_judge,eval_score_LLM_judge_int
0,What architecture is the `tokenizers-linux-x64...,x86_64-unknown-linux-musl,huggingface/tokenizers/blob/main/bindings/node...,The `tokenizers-linux-x64-musl` binary is desi...,3,The response accurately identifies the archite...,1.0
1,What is the purpose of the BLIP-Diffusion mode...,The BLIP-Diffusion model is designed for contr...,huggingface/diffusers/blob/main/docs/source/en...,"The BLIP-Diffusion model, as described in Docu...",3,The response accurately describes the purpose ...,1.0
2,How can a user claim authorship of a paper on ...,By clicking their name on the corresponding Pa...,huggingface/hub-docs/blob/main/docs/hub/paper-...,To claim authorship of a paper on the Hugging ...,3,The response accurately describes the process ...,1.0
3,What is the purpose of the /healthcheck endpoi...,Ensure the app is running,huggingface/datasets-server/blob/main/services...,The purpose of the `/healthcheck` endpoint in ...,3,The response accurately and completely describ...,1.0
4,What is the default context window size for Lo...,127 tokens,huggingface/transformers/blob/main/docs/source...,The default context window size for Local Atte...,3,The response accurately identifies the default...,1.0
...,...,...,...,...,...,...,...
60,What is the maximum size of a model checkpoint...,10GB,huggingface/transformers/blob/main/docs/source...,The maximum size of a model checkpoint before ...,3,The response accurately identifies the maximum...,1.0
61,What is the purpose of Weights and Biases (W&B...,To track their machine learning experiments at...,gradio-app/gradio/blob/main/guides/06_integrat...,Weights and Biases (W&B) is a platform designe...,3,The response accurately describes the purpose ...,1.0
62,What is the name of the open-source library cr...,Optimum,huggingface/blog/blob/main/intel.md,The name of the open-source library created by...,1,"The response incorrectly identifies ""Accelerat...",0.0
63,What parameter is used to ensure that elements...,equal_height,gradio-app/gradio/blob/main/guides/cn/03_build...,The parameter used to ensure that elements in ...,3,The response includes the correct parameter na...,1.0


In [67]:
# save all results

# Create a dictionary of JSON strings
json_results = {}
results_dir = "results"

for system_type, df in results.items():
    json_results[system_type] = json.loads(df.to_json(orient='records'))
save_json_path = os.path.join(results_dir, model_name + "_vect400_t0.2.json")

# Save to a file
with open(save_json_path, 'w') as f:
    json.dump(json_results, f, indent=4)

In [ ]:
# load results

results_dir = "results"
load_model_name = ""
load_json_path = os.path.join(results_dir,
                              load_model_name + "_vect400_t0.2.json")

# Read the JSON file
with open('load_json_path', 'r') as f:
    loaded_json = json.load(f)

# Convert back to a dictionary of DataFrames
loaded_results = {}
for system_type, records in loaded_json.items():
    loaded_results[system_type] = pd.DataFrame.from_records(records)

results = loaded_results